# T1-09 Prompt vs RAG

**Time:** 12:45–13:45 &nbsp;|&nbsp; **Evidence:** side-by-side log — invented circular vs cited paragraph vs abstain

**Notebook learning outcomes**

- Show the **same question** answered from weights only vs from a retrieved chunk.
- Name **hallucination** when a circular number appears with no source.
- **Abstain** when the corpus has no paragraph (Q-MISSING).


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers

**Prompt-only** here means: the generator never sees a handbook chunk. It continues from the standing instruction plus the user text. That is Day 1.

**RAG** means: retrieve first, then generate from visible chunks.

The failure mode ministries actually fear is not a blank. It is a **fluent wrong**: “CivicDesk Circular 99 of 2024 appears to allow…” — invented, confident, unusable in Finance or Defence.

![Same question, two paths](images/t2_prompt_vs_rag.jpg)

**Extra words:**

- **Grounded** — every factual clause points at a chunk id.
- **Ungrounded / hallucinated** — the clause cannot be found in the prompt or the retrieved set.
- **Abstain** — the honest third output. Not a shrug: a logged stop.


## Ministry link

Finance / Defence / Health cannot accept an invented circular, OPR note, or guideline number. Labour cannot accept an invented C2. Trade cannot accept “your password has been reset” because the model sounded sure.

This lab is **N3**. The abstain on Q-MISSING is the skill to take home.


## 1. Same question, two replies


In [ ]:
# [RUN]
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}["Q-MISSING"]
print("USER:", q["user"])
print()
print("--- prompt-only ---")
po = rag.generate_prompt_only(q["user"])
print("evidence:", po["evidence"], "mode:", po["mode"])
print(po["reply"])
print()
print("--- RAG ---")
out = rag.generate_with_rag(q["user"], role="clerk")
print("evidence:", out["evidence"], "mode:", out["mode"], "visible:", out["visible_ids"])
print(out["reply"])
if out["mode"] != "live":
    print("not a model answer, not scored")
rag.save_log("T1-09_missing", {"prompt_only": po, "rag": {k: out[k] for k in out if k != "visible_chunks"}})


### Teaching note

Mark a prompt-only circular number with no source as a fail, even if it reads fluently.

Without a matching **RECORDED MODEL RUN**, leave model-performance comparisons unmeasured. Local template continuation still shows the pipeline.


## Run 1: the frozen set

Twelve items. Live only the ids you list. Everything else comes from a replay file if one matches; otherwise **model result unavailable**. Simulations never enter model accuracy totals.


In [ ]:
# [CHANGE] pick two or three ids to attempt live. Leave the rest to replay or unavailable.
LIVE_IDS = ["Q-ATTACH", "Q-MISSING"]
CONFIG_1 = {"retriever": "keyword", "min_score": 2, "current_only": False, "system": "STRICT_SYSTEM", "k": 3}

import civicdesk_rag as rag

run1 = rag.run_known_answer_set(CONFIG_1, live_ids=LIVE_IDS, replay="T1-09_run1.json")
rag.save_log("T1-09_run1_local", {"config": CONFIG_1, "totals": run1["totals"]})


## Run 2: change one thing

Change **exactly one** of `min_score`, `current_only`, or the system text. Prepared replay names: `T1-09_run2_current_only.json`, `T1-09_run2_min_score.json`.


In [ ]:
# [CHANGE] exactly one key vs CONFIG_1.
CONFIG_2 = {"retriever": "keyword", "min_score": 2, "current_only": True, "system": "STRICT_SYSTEM", "k": 3}
REPLAY_2 = "T1-09_run2_current_only.json"

import civicdesk_rag as rag

run2 = rag.run_known_answer_set(CONFIG_2, live_ids=[], replay=REPLAY_2)
changed = rag.compare_runs(run1, run2)
stale = rag.load_replay("T1-09_run1.json")
print("stale run1 replay usable under CONFIG_2:", rag.replay_usable(stale, CONFIG_2) if stale else "no replay file (model result unavailable)")
if stale and rag.replay_usable(stale, CONFIG_1) and not rag.replay_usable(stale, CONFIG_2):
    print("stale replay rejected after the setting change")


## Three failures

Noise-hit / paraphrase, stale-rule, Circular 88 false-premise. Write the failing stage: retrieval, source, or generation.


In [ ]:
# [RUN]
import civicdesk_rag as rag

ka = {i["id"]: i for i in rag.load_known_answer_set()["items"]}
for iid in ("Q-PARAPHRASE", "Q-STALE", "Q-FALSE-PREMISE"):
    item = ka[iid]
    print("=" * 60, iid)
    print("user:", item["user"])
    print("expected:", item["expected_outcome"], "stage_if_wrong:", item["stage_if_wrong"])
    print("why:", item["why"])
    print("hits:", [(h["id"], h["score"]) for h in rag.retrieve(item["user"], k=5, current_only=False)])


Your notes: paraphrase failing stage ____ ; stale-rule ____ ; Circular 88 ____


In [ ]:
# [CHANGE] Mark two cite items by hand. yes / partly / no.
SUPPORTED = {"Q-ATTACH": None, "Q-C2": None}

auto_cite_ok = sum(1 for r in run1["rows"] if r.get("expected") == "cite" and r.get("met"))
hand = sum(1 for v in SUPPORTED.values() if str(v).lower() == "yes")
print("automatic cite-met (live+recorded only, and only if scored):", auto_cite_ok)
print("hand-marked supported yes:", hand, "of", len(SUPPORTED))
print("A cite outcome still needs a person to judge whether the quote supports the answer.")


In [ ]:
# [OPTIONAL] cloud baseline. Fictional items only. Free tiers may keep prompts.
print("Optional. Reuse the T1-03 comparison key if you have a live cloud endpoint. Not run in the default room.")
print("If you run it, print totals next to OLMo and label the provider. Do not paste ministry text.")


### After you run

Use the **counts** (met / not met / unavailable / simulated). Do not declare a production verdict from this classroom set. A mixed batch is not a single live benchmark.

**Write-up:** If the chunk is not in the retrieved set, we will not invent a circular.
